# PEAK M-ATH — Identifying Malicious Base64 Encoded Payloads in Scripts

Detect and decode Base64 commands in EDR logs; flag unusually long commands and obfuscation patterns, following the **PEAK** framework: **Prepare → Execute → Act → Knowledge**.

**Ref:** M05 — Identifying malicious base64 encoded payloads in scripts  
**M-ATH approach:** Regex-based Base64 extraction, decoding, and heuristic scoring (length, entropy, suspicious decoded content).

## How to use
1. Put EDR CSV files into `input/` (with command-line columns, e.g. src.process.commandLine, src.process.args)
2. Run all cells
3. Review findings in `output/`

In [ ]:
pass  # Placeholder (removed environment-specific output)

## PREPARE — Plan your Approach

- **Select Topic:** Malicious Base64 payloads in scripts — adversaries encode commands to evade signature-based detection (MITRE ATT&CK [T1027](https://attack.mitre.org/techniques/T1027/) Obfuscated Files or Information, [T1059.001](https://attack.mitre.org/techniques/T1059/001/) PowerShell).
- **Research Topic:** Base64 encoding patterns in PowerShell and script interpreters, common obfuscation techniques, decoded-content indicators.
- **Identify Datasets:** EDR logs with command-line columns (e.g. `src.process.commandLine`, `src.process.args`).
- **Select Algorithms:** Regex-based Base64 extraction, decoding with error handling, heuristic scoring (encoded length, decoded entropy, suspicious content patterns).

In [ ]:
# Scenario mode: always prefer the scenario-local input/output folders
import os
import sys
from pathlib import Path

REPO_CANDIDATE = Path('.').resolve()
SCENARIO_REL = Path('scenarios') / 'identifying_malicious_base64_encoded_payloads_in_scripts'
if (REPO_CANDIDATE / SCENARIO_REL).exists():
    REPO_ROOT = REPO_CANDIDATE
    SCENARIO_DIR = (REPO_ROOT / SCENARIO_REL).resolve()
else:
    SCENARIO_DIR = Path('.').resolve()
    REPO_ROOT = SCENARIO_DIR.parent.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

INPUT_DIR = SCENARIO_DIR / 'input'
OUTPUT_DIR = SCENARIO_DIR / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SCENARIO_MODE = True

In [ ]:
import base64
import glob
import math
import re
from pathlib import Path

import pandas as pd

pd.set_option('display.max_colwidth', 180)

if not globals().get('SCENARIO_MODE', False):
    INPUT_DIR = Path('input')
    OUTPUT_DIR = Path('output')
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def _rel(p):
    try:
        if globals().get('SCENARIO_MODE', False) and 'REPO_ROOT' in globals() and hasattr(p, 'is_relative_to') and p.is_relative_to(REPO_ROOT):
            return p.relative_to(REPO_ROOT)
        return p
    except (ValueError, AttributeError):
        return p

print(f'Input folder: {_rel(INPUT_DIR)}')
print(f'Output folder: {_rel(OUTPUT_DIR)}')

## Base64 decode and scoring helpers

In [ ]:
def shannon_entropy(text: str) -> float:
    text = (text or '').strip()
    if not text:
        return 0.0
    probs = [text.count(ch) / len(text) for ch in set(text)]
    return -sum(p * math.log2(p) for p in probs)


def try_decode_base64_token(token: str) -> str:
    cleaned = token.strip()
    if len(cleaned) < 12:
        return ''
    if not re.fullmatch(r'[A-Za-z0-9_+/=-]+', cleaned):
        return ''
    normalized = cleaned.replace('-', '+').replace('_', '/')
    normalized = normalized.split('=')[0]
    padded = normalized + '=' * ((4 - len(normalized) % 4) % 4)
    try:
        raw = base64.b64decode(padded, validate=True)
        decoded = raw.decode('utf-8')
    except Exception:
        return ''
    printable_ratio = sum(ch.isprintable() for ch in decoded) / max(len(decoded), 1)
    has_letters = any(ch.isalpha() for ch in decoded)
    if printable_ratio < 0.9 or not has_letters:
        return ''
    return decoded.strip()


def extract_base64_findings(text: str) -> list[dict]:
    """Extract all decodable Base64 tokens and return findings with score/reasons."""
    text = str(text or '').strip()
    if not text:
        return []
    candidates = re.findall(r'[A-Za-z0-9_+/=-]{12,}', text)
    seen = set()
    findings = []
    for token in candidates:
        if token in seen:
            continue
        seen.add(token)
        decoded = try_decode_base64_token(token)
        if not decoded:
            continue
        score = 0
        reasons = []
        if len(token) >= 100:
            score += 3
            reasons.append('very-long-base64')
        elif len(token) >= 50:
            score += 2
            reasons.append('long-base64')
        else:
            score += 1
            reasons.append('base64-encoded')
        dec_lower = decoded.lower()
        if 'powershell' in dec_lower or 'pwsh' in dec_lower:
            score += 2
            reasons.append('decoded-powershell')
        if 'cmd' in dec_lower and ('/c' in dec_lower or '/k' in dec_lower):
            score += 2
            reasons.append('decoded-cmd')
        if 'http://' in dec_lower or 'https://' in dec_lower:
            score += 1
            reasons.append('decoded-url')
        if 'invoke-' in dec_lower or 'iex' in dec_lower or 'downloadstring' in dec_lower:
            score += 2
            reasons.append('decoded-suspicious-cmdlet')
        entropy = shannon_entropy(decoded)
        if entropy > 4.0:
            score += 1
            reasons.append('high-entropy-decoded')
        findings.append({
            'token_len': len(token),
            'decoded': decoded[:500],
            'decoded_len': len(decoded),
            'score': score,
            'reasons': ','.join(reasons)
        })
    return findings

## EXECUTE — Experimentation Time

- **Gather Data:** Load EDR CSVs from `input/`, detect command-line columns.
- **Pre-Process Data:** Concatenate sources, identify rows with potential Base64 content.
- **Apply:** Extract Base64 strings, decode them, score findings by length, entropy, and decoded-content indicators.
- **Analyze:** Review scored findings sorted by risk; inspect decoded payloads for malicious intent.
- **Escalate Critical Findings:** Decoded payloads with clear malicious content (download cradles, shellcode, credential access) warrant immediate incident response.

## Load EDR data

In [ ]:
scenario_name = 'identifying_malicious_base64_encoded_payloads_in_scripts'
candidate_dirs = []

if 'SCENARIO_DIR' in globals():
    candidate_dirs.append(Path(SCENARIO_DIR) / 'input')
if 'REPO_ROOT' in globals():
    candidate_dirs.append(Path(REPO_ROOT) / 'scenarios' / scenario_name / 'input')
    candidate_dirs.append(Path(REPO_ROOT) / 'input')

cwd = Path.cwd()
candidate_dirs.append(cwd / 'input')
candidate_dirs.append(cwd / 'scenarios' / scenario_name / 'input')
for parent in cwd.parents:
    candidate_dirs.append(parent / 'scenarios' / scenario_name / 'input')

# Fallback: discover scenario input recursively from the current working tree.
candidate_dirs.extend(cwd.glob(f'**/scenarios/{scenario_name}/input'))

# Keep order, drop duplicates, and only keep existing directories
seen = set()
search_dirs = []
for d in candidate_dirs:
    resolved = d.resolve()
    key = str(resolved).lower()
    if key in seen:
        continue
    seen.add(key)
    if resolved.exists() and resolved.is_dir():
        search_dirs.append(resolved)

csv_paths = []
for d in search_dirs:
    csv_paths.extend(str(p) for p in d.rglob('*') if p.is_file() and p.suffix.lower() == '.csv')
csv_paths = sorted(set(csv_paths))

print(f'Searching CSVs in: {[_rel(d) for d in search_dirs]}')
print(f'Found {len(csv_paths)} CSV file(s).')

if not csv_paths:
    raise FileNotFoundError(
        'No CSV files found in discovered input folders. '
        f'Searched: {[_rel(d) for d in search_dirs]}'
    )

dfs = []
for p in csv_paths:
    df = pd.read_csv(p)
    try:
        src_rel = str(Path(p).relative_to(REPO_ROOT)) if 'REPO_ROOT' in globals() and Path(p).is_relative_to(REPO_ROOT) else p
    except (ValueError, AttributeError):
        src_rel = p
    df['_source_file'] = src_rel
    dfs.append(df)
raw = pd.concat(dfs, ignore_index=True)

scan_cols = [
    c for c in raw.columns
    if (
        'command' in c.lower()
        or 'cmdline' in c.lower()
        or 'args' in c.lower()
        or c == 'commandLine'
        or 'activecontent' in c.lower()
        or 'cmdscript' in c.lower()
    )
]
if not scan_cols:
    scan_cols = [raw.columns[0]] if len(raw.columns) > 0 else []
if not scan_cols:
    raise ValueError(f'No text columns found to scan. Columns: {list(raw.columns)}')

for c in scan_cols:
    raw[c] = raw[c].fillna('').astype(str).str.strip()

raw = raw[raw[scan_cols].apply(lambda s: s.str.len() > 0).any(axis=1)].copy()
print(f"Loaded {len(raw):,} rows with data in scan columns: {scan_cols}")

## Extract and score Base64 findings

In [ ]:
all_findings = []
for idx, row in raw.iterrows():
    for col in scan_cols:
        value = str(row.get(col, '')).strip()
        if not value:
            continue
        findings = extract_base64_findings(value)
        for f in findings:
            all_findings.append({
                'row_index': idx,
                'source_file': row.get('_source_file', ''),
                'source_column': col,
                'command_preview': value[:300],
                **f
            })

findings_df = pd.DataFrame(all_findings)
if len(findings_df) == 0:
    print('No decodable Base64 findings.')
else:
    findings_df = findings_df.sort_values(['score', 'token_len'], ascending=False).reset_index(drop=True)
    print(f"Findings: {len(findings_df):,}")
    display(findings_df.head(20))

## ACT — Wrapping Up the Investigation

- **Document Findings:** Scored Base64 findings with decoded content exported for analyst triage.
- **Preserve Hunt:** Results archived to `output/base64_payloads_results.csv`.
- **Create Detections / Playbooks:** Confirmed malicious patterns can feed PowerShell constrained language mode policies or AMSI-based detection rules.

In [ ]:
out_path = OUTPUT_DIR / 'base64_payloads_results.csv'
if len(findings_df) > 0:
    findings_df.to_csv(out_path, index=False)
    print(f'Saved to {_rel(out_path)}')
else:
    pd.DataFrame(
        columns=[
            'row_index',
            'source_file',
            'source_column',
            'command_preview',
            'token_len',
            'decoded',
            'decoded_len',
            'score',
            'reasons'
        ]
    ).to_csv(out_path, index=False)
    print(f'Empty results saved to {_rel(out_path)}')

## KNOWLEDGE — Continuous Improvement

- **Re-Add Topics to Backlog:** New obfuscation techniques or script interpreters identified during analysis become future hunt topics.
- **Communicate Findings:** Share confirmed malicious payloads with SOC, endpoint security teams, and threat intelligence for signature creation.
- **Feed Improvements Back:** Update scoring heuristics based on confirmed true/false positives; add new decoded-content indicators as attack patterns evolve.
- **Measure Effectiveness:** Track detection rates and confirmed malicious payloads across successive runs to assess hunting value.